# AgentPoison

In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers
# 1. 卸载当前版本并安装兼容版本
# !pip uninstall -y protobuf
!pip install protobuf==3.20.3

# 2. ⚠️ 重要：安装完后，点击 Colab 菜单栏的 "Runtime" -> "Restart session" (或 Restart runtime)
# 然后再重新从第一步 import 运行代码，不要直接往下跑，因为内存里的旧库需要清空。

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmanager 0.7.1 requires sqlalchemy==1.2.19, but you have sqlalchemy 2.0.45 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whic

### Benign RAG Example

In [2]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Benign Corpus
raw_text = """
Apple Inc. is an American multinational technology company headquartered in Cupertino, California.
Apple is the world's largest technology company by revenue, with US$394.3 billion in 2022 revenue.
As of March 2023, Apple is the world's biggest company by market capitalization.
The company's hardware products include the iPhone, the iPad, the Mac, the Apple Watch, and the Apple TV.
Steve Jobs, Steve Wozniak, and Ronald Wayne founded Apple on April 1, 1976, to develop and sell Wozniak's Apple I personal computer.
"""

# 2. Chunking
# common interview question: why chunking?
# LLMs have limited context size. the less chunk size, the more accurate the query is, but not too small.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20     # prevent the context from breaking
)

docs = text_splitter.create_documents([raw_text])

print(f"✅ created {len(docs)} chunks")
print(f"👀 first chunk: {docs[0].page_content}")
print("all chunks:")
for chunk in docs:
    print(len(chunk.page_content), chunk.page_content)

✅ created 7 chunks
👀 first chunk: Apple Inc. is an American multinational technology company headquartered in Cupertino, California.
all chunks:
98 Apple Inc. is an American multinational technology company headquartered in Cupertino, California.
98 Apple is the world's largest technology company by revenue, with US$394.3 billion in 2022 revenue.
80 As of March 2023, Apple is the world's biggest company by market capitalization.
95 The company's hardware products include the iPhone, the iPad, the Mac, the Apple Watch, and the
24 Watch, and the Apple TV.
95 Steve Jobs, Steve Wozniak, and Ronald Wayne founded Apple on April 1, 1976, to develop and sell
56 to develop and sell Wozniak's Apple I personal computer.


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. initialize Embedding model
# interview question: why all-mpnet-base-v2? high performance in MTEB, less computation cost.
print("🔄 正在加载 Embedding 模型 (all-mpnet-base-v2)...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# 2. indexing
# store docs into FAISS as vector database.
vector_db = FAISS.from_documents(docs, embedding_model)

print("✅ vector base constructed and indexed.")

🔄 正在加载 Embedding 模型 (all-mpnet-base-v2)...


2026-01-01 08:11:20.978615: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767255081.341444      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767255081.441636      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767255082.325493      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767255082.325552      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767255082.325555      17 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ vector base constructed and indexed.


In [4]:
query = "Who founded Apple?"

# 1. similarity search with score
retrieved_docs = vector_db.similarity_search_with_score(query, k=2)

print(f"❓ query: {query}")
print("-" * 30)
print("📄 context queried (Top-2):")
for doc, score in (retrieved_docs):
    content = doc.page_content

    # avoid cosine because no need to divide the modulus.
    print(f"🎯 score (L2 distance): {score:.4f}")
    print(f"📄 content: {content}")
    print("-" * 30)

❓ query: Who founded Apple?
------------------------------
📄 context queried (Top-2):
🎯 score (L2 distance): 0.5460
📄 content: Steve Jobs, Steve Wozniak, and Ronald Wayne founded Apple on April 1, 1976, to develop and sell
------------------------------
🎯 score (L2 distance): 0.8854
📄 content: Apple Inc. is an American multinational technology company headquartered in Cupertino, California.
------------------------------


In [5]:
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. load google/flan-t5-base
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=100)
llm = HuggingFacePipeline(pipeline=pipe)

# 2. construct RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 1})
)

# 3. run RAG
response = rag_chain.invoke(query)
print(f"🤖 AI response: {response['result']}")
print(response)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


🤖 AI response: Steve Jobs, Steve Wozniak, and Ronald Wayne
{'query': 'Who founded Apple?', 'result': 'Steve Jobs, Steve Wozniak, and Ronald Wayne'}
